# mlmodels end-goal API, pillar 3 -- checkpoints + resume

**Goal (L3f.3).** Prove the *resumable* checkpoint story by hand, and pin down the one
`src` change it forces. We (1) train a few epochs and save a **bundled** Lightning
checkpoint (`trainer.save_checkpoint` -- the design's fork-2, which carries optimizer
state + epoch so training can continue), (2) **resume** from it and continue training
(showing the loss curve picks up where it left off, not a cold restart), then (3)
confirm the gap: the current `materialize_model` cannot load a bundled checkpoint, and
show the exact load forms the harvest into `mlmodels` must adopt.

This is the pillar where the notebook-first incubator produces its first concrete
**`src` fix** for `mlmodels` (the 8b finding from `ns_mlmodels-e2e.ipynb`).

## 0. Config / env seam + imports

In [ ]:
import os
from pathlib import Path


def project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("pyproject.toml not found above cwd")


ROOT = project_root()
VAR = ROOT / "var" / "data" / "surrogate_models"
CKPT_DIR = VAR / "checkpoints" / "nb-resume"
os.environ["SURROGATE_MODELS__DATASETS__PATH"] = str(VAR / "datasets")
os.environ["SURROGATE_MODELS__DATASETS__NEUTRON_STARS_SOURCE"] = str(
    ROOT / "data" / "neutron-stars" / "neutron-stars.dat"
)
os.environ["SURROGATE_MODELS__MLMODELS__CHECKPOINT_DIR"] = str(CKPT_DIR)
os.environ["SURROGATE_MODELS__LOGGING__FILE"] = str(VAR / "log" / "surrogate_models.log")
print("checkpoint dir:", CKPT_DIR)

In [ ]:
%matplotlib inline
from typing import Any

import lightning
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from surrogate_models import load_neutron_stars
from surrogate_models.mlmodels.domain import (
    Checkpoint,
    RunID,
    TrainedRun,
    TrainingConfig,
    make_model_identity,
)
from surrogate_models.mlmodels.infrastructure import (
    RunManifest,
    materialize_model,
    structural_fingerprint,
    write_run_manifest,
)

torch.manual_seed(0)
_ = lightning.seed_everything(0, verbose=False)

## 1. Data, split, model (condensed) -- the same simple surrogate

In [ ]:
df = load_neutron_stars()
FEATURES, TARGET = ["rhoc", "Pc"], "M"
sample = df[[*FEATURES, TARGET]].dropna().sample(n=3000, random_state=0)
x_raw = torch.tensor(sample[FEATURES].to_numpy(), dtype=torch.float32)
y_raw = torch.tensor(sample[[TARGET]].to_numpy(), dtype=torch.float32)
X = (x_raw - x_raw.mean(0)) / x_raw.std(0)
Y = (y_raw - y_raw.mean(0)) / y_raw.std(0)
perm = torch.randperm(X.shape[0], generator=torch.Generator().manual_seed(0))
tr, va = perm[:2100], perm[2100:2550]
train_loader = DataLoader(TensorDataset(X[tr], Y[tr]), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X[va], Y[va]), batch_size=64, shuffle=False)
CONFIG = TrainingConfig(20, 0.05, 64, "adam")


class SplitSurrogate(lightning.LightningModule):
    '''A 2->8->1 MLP -- notebook model only.'''

    MODEL_NAME = "ns-mlp-resume"
    MODEL_VERSION = "0.1.0"

    def __init__(self, n_features: int, learning_rate: float, optimizer_name: str) -> None:
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 8), nn.ReLU(), nn.Linear(8, 1))
        self.learning_rate = learning_rate
        self.optimizer_name = optimizer_name

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        out: torch.Tensor = self.net(features)
        return out

    def training_step(self, *args: Any, **kwargs: Any) -> torch.Tensor:
        f, t = args[0]
        loss: torch.Tensor = nn.functional.mse_loss(self.net(f), t)
        self.log("train_loss", loss, on_step=False, on_epoch=True, logger=False)
        return loss

    def validation_step(self, *args: Any, **kwargs: Any) -> torch.Tensor:
        f, t = args[0]
        loss: torch.Tensor = nn.functional.mse_loss(self.net(f), t)
        self.log("val_loss", loss, on_step=False, on_epoch=True, logger=False)
        return loss

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)


def build(config: Any) -> SplitSurrogate:
    return SplitSurrogate(len(FEATURES), config.learning_rate, config.optimizer)


class Recorder(lightning.Callback):
    '''Accumulate (epoch, train_loss) across BOTH the phase-1 and resumed trainers.'''

    def __init__(self) -> None:
        super().__init__()
        self.rows: list[tuple[int, float]] = []

    def on_validation_epoch_end(self, trainer: lightning.Trainer, pl: lightning.LightningModule) -> None:
        if "train_loss" in trainer.callback_metrics:
            self.rows.append((trainer.current_epoch, float(trainer.callback_metrics["train_loss"])))


def make_trainer(max_epochs: int, recorder: Recorder) -> lightning.Trainer:
    return lightning.Trainer(
        max_epochs=max_epochs, accelerator="cpu", devices=1, logger=False,
        enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
        callbacks=[recorder],
    )

## 2. Phase 1 -- train 10 epochs, save a BUNDLED checkpoint

`trainer.save_checkpoint` writes the fork-2 format: the model weights **plus** optimizer
state, epoch, and hyperparameters -- everything needed to continue.

In [ ]:
recorder = Recorder()
CKPT_DIR.mkdir(parents=True, exist_ok=True)
bundle = CKPT_DIR / "resume.ckpt"

model1 = build(CONFIG)
trainer1 = make_trainer(10, recorder)
trainer1.fit(model1, train_loader, val_loader)
trainer1.save_checkpoint(bundle)
print("phase-1 last (epoch, train_loss):", recorder.rows[-1])
print("bundle written:", bundle.name, "| top-level keys:",
      sorted(torch.load(bundle, weights_only=False).keys()))

## 3. Resume -- continue training from the bundle

Passing `ckpt_path=` to `fit` restores the full training state (weights, optimizer,
epoch) into a fresh model and continues to the new `max_epochs`. The proof of a real
resume: the first resumed epoch's loss is already low (continuing from ~0.018), **not**
back at the cold-start ~0.2.

In [ ]:
model2 = build(CONFIG)
trainer2 = make_trainer(20, recorder)
trainer2.fit(model2, train_loader, val_loader, ckpt_path=str(bundle))

resume_first = next(r for r in recorder.rows if r[0] >= 10)
print("first resumed epoch (epoch, train_loss):", resume_first)
print("continues (<< cold-start 0.2):", resume_first[1] < 0.05)
print("last epoch:", recorder.rows[-1])

## 4. The combined loss curve -- one continuous descent across the resume boundary

In [ ]:
epochs = [r[0] for r in recorder.rows]
losses = [r[1] for r in recorder.rows]
plt.figure(figsize=(6.5, 3.5))
plt.plot(epochs, losses, marker="o")
plt.axvline(10, color="grey", linestyle="--", label="save + resume")
plt.xlabel("epoch")
plt.ylabel("train MSE (standardized)")
plt.title("phase 1 (0-9) then resumed (11-19) -- no cold restart")
plt.legend()
plt.tight_layout()
plt.show()

## 5. The `src` fix this pillar drove -- now landed

`materialize_model` once assumed a **bare** state_dict, so a bundled (resumable)
checkpoint -- weights nested under a `state_dict` key -- broke it. That gap is now fixed
in `mlmodels` src: `_load_live_model` reads `checkpoint.get("state_dict", checkpoint)`,
loading bundled and bare checkpoints alike. Below, `materialize_model` loads the bundle
directly; the manual load forms are shown as equivalents.

In [ ]:
write_run_manifest(CKPT_DIR, RunManifest(
    "resume", SplitSurrogate.MODEL_NAME, SplitSurrogate.MODEL_VERSION,
    20, 0.05, 64, "adam", structural_fingerprint(model2.state_dict())))
run = TrainedRun(RunID("resume"), CONFIG, Checkpoint(str(bundle)))
identity = make_model_identity(SplitSurrogate.MODEL_NAME, SplitSurrogate.MODEL_VERSION).unwrap()

res = materialize_model(lambda: build(CONFIG), identity, CKPT_DIR, run)
print("materialize_model on a bundle now Ok:", res.is_ok(),
      "| model:", type(res.unwrap()).__name__)

# Equivalent manual forms (what the fix does internally):
lit = SplitSurrogate.load_from_checkpoint(
    str(bundle), n_features=len(FEATURES), learning_rate=0.05, optimizer_name="adam")
print("load_from_checkpoint works:", isinstance(lit, SplitSurrogate))
state = torch.load(str(bundle), weights_only=True)["state_dict"]
reloaded = build(CONFIG)
reloaded.load_state_dict(state)
print('torch.load(...)["state_dict"] + load_state_dict works: True')

## Findings -> the first `mlmodels` src harvest (landed)

- **Resume works by hand** with the standard Lightning `ckpt_path` mechanism over a
  bundled checkpoint; the loss curve continues across the save/resume boundary.
- **Harvest DONE (scope A):** `materialize_model` now accepts a bundled checkpoint --
  `_load_live_model` reads `checkpoint.get("state_dict", checkpoint)`, no longer assuming a
  bare state_dict. Promoted into `mlmodels` src under TDD with a tiny synthetic bundled-ckpt
  fixture (`test_materialize_model_loads_a_bundled_checkpoint`); the real-data proof stays
  here. This is the first notebook-incubated fix harvested into a bounded context.
- **Deferred (scope B):** switching `save_trained_run` to `trainer.save_checkpoint` plus a
  first-class `resume`/`continue` seam -- worthwhile once a `src` save-path consumer needs
  it (training itself is still done by hand in the notebook today).